# DRISHTI MVP on Kaggle
GPU: T4 / P100. Add CUHK-PEDES as a Kaggle dataset (it must be obtained from its authors) and set the paths below.

In [ ]:
!git clone <your-repo-url> drishti
%cd drishti
!pip install -q -r requirements.txt
!python -m pytest -q

In [ ]:
DATA = '/kaggle/input/cuhk-pedes'   # folder with reid_raw.json and imgs/
OPTS = f'data.json={DATA}/reid_raw.json data.image_root={DATA}/imgs output_root=/kaggle/working/outputs'

In [ ]:
!python scripts/make_noisy_sets.py --config configs/mvp_baseline.yaml --out query_sets/cuhk_pedes --opts $OPTS

In [ ]:
!python eval.py --config configs/mvp_baseline.yaml --query-sets 'query_sets/cuhk_pedes/*.json' --results /kaggle/working/outputs/results.csv --opts $OPTS

In [ ]:
SEED = 1
!python train.py --config configs/mvp_baseline.yaml --seed $SEED --opts $OPTS
!python train.py --config configs/mvp_robust.yaml --seed $SEED --opts $OPTS

In [ ]:
for run in [f'mvp_baseline_seed{SEED}', f'mvp_robust_seed{SEED}']:
    !python eval.py --config configs/mvp_baseline.yaml --checkpoint /kaggle/working/outputs/{run}/best.pt --query-sets 'query_sets/cuhk_pedes/*.json' --results /kaggle/working/outputs/results.csv --save-gallery /kaggle/working/outputs/{run}/gallery.npz --opts $OPTS

In [ ]:
!python scripts/summarise_results.py --results /kaggle/working/outputs/results.csv --out /kaggle/working/outputs/summary.md